# Comparação de Campanhas — RMI vs Planilha SMA

Compara três campanhas de disparo WhatsApp enviadas para servidores municipais:

| Campanha | Template HSM | Fonte da base | Data disparo |
|---|---|---|---|
| Dia do Servidor (RMI) | `recado_dia_do_servidor_v3` | RMI (pessoa_fisica) | 2026-10-28 |
| MobiRio (SMA) | `recado_mobirio_pref_rio_bonus_v1` | Planilha SMA | 2026-10-26 |
| Servidores 2º envio (SMA) | `recado_servidores_pref_rio_14` | Planilha SMA | 2026-10-26 |

**Diferença principal entre as fontes:**
- **RMI**: base construída a partir do `rmi_dados_mestres.pessoa_fisica` — telefones validados e priorizados pelo algoritmo do RMI (estratégia de envio, qualidade do telefone, deduplica por CPF).
- **Planilha SMA**: base fornecida pela Secretaria Municipal de Administração, com telefones do sistema Ergon. A validação é feita pela UDF `VALIDATE_AND_FORMAT_PHONE` na query de disparo, mas sem priorização de qualidade.

**Seções:**
1. Funil de envio por campanha (disparados → entregues → lidos → respondidos)
2. Falhas: decomposição por categorias mutuamente exclusivas
3. Top erros Meta por campanha
4. Sobreposição de telefones entre as campanhas
5. Sobreposição de CPFs entre as campanhas
6. Tabela resumo comparativa

---
**Notas sobre métricas de entrega/leitura:**  
Usamos `entrega_confirmada_indicador` e `leitura_confirmada_indicador` (eventos OttDelivered/OttRead observados diretamente),  
**não** `foi_entregue`/`foi_lida` — essas inferem entrega/leitura a partir de resposta pareada e superestimam (~10pp e ~18pp respectivamente).

In [1]:
import os
import pandas as pd
import pandas_gbq
from google.oauth2 import service_account
from google.cloud import bigquery

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

# ------------------------------------------------------------------
# Credenciais
# As tabelas bonus_mobi_rio e bonus_servidor_segundo_envio são
# tabelas EXTERNAS vinculadas ao Google Sheets. Por isso precisam
# do scope drive.readonly, que o basedosdados não inclui.
# Usamos a service account do dbt com esse scope adicionado.
# ------------------------------------------------------------------
KEYFILE   = os.path.expanduser("~/.dbt/tmp/credentials.json")
SCOPES    = [
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/cloud-platform",
    "https://www.googleapis.com/auth/drive.readonly",
]
CREDENTIALS     = service_account.Credentials.from_service_account_file(KEYFILE, scopes=SCOPES)
BILLING_PROJECT = "rj-crm-registry"

bq = bigquery.Client(project=BILLING_PROJECT, credentials=CREDENTIALS)

def run_query(sql: str) -> pd.DataFrame:
    return pandas_gbq.read_gbq(
        sql,
        project_id=BILLING_PROJECT,
        credentials=CREDENTIALS,
        progress_bar_type=None,
    )

# Templates HSM
HSM_RMI        = "recado_dia_do_servidor_v3"
HSM_MOBI_RIO   = "recado_mobirio_pref_rio_bonus_v1"
HSM_SERVIDORES = "recado_servidores_pref_rio_14"

# Tabelas
TABELA_MOBI_RIO      = "`rj-crm-registry-dev.teste.bonus_mobi_rio`"
TABELA_SERVIDORES    = "`rj-crm-registry-dev.teste.bonus_servidor_segundo_envio`"
FCT_CHATBOT          = "`rj-crm-registry.rmi_conversas.chatbot`"
TABELA_PESSOA_FISICA = "`rj-crm-registry.rmi_dados_mestres.pessoa_fisica`"


print("Setup concluído. Credenciais carregadas de:", KEYFILE)

/home/patricia/Documentos/Work/escritorio_dados/esc_venv_310/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.4.3)/charset_normalizer (3.4.0) doesn't match a supported version!
  warnings.warn(


Setup concluído. Credenciais carregadas de: /home/patricia/.dbt/tmp/credentials.json


## 1. Funil de envio por campanha

Conta telefones únicos em cada etapa do ciclo de entrega para as três campanhas.

- **Dia do Servidor (RMI)**: disparo em 2026-10-28
- **MobiRio e Servidores 2º envio (SMA)**: disparo em 2026-10-26

Métricas de falha separadas:
- `erro_salesforce`: falha na plataforma antes de chegar à Meta
- `erro_meta`: erro retornado pela Meta (qualquer código)
- `erro_meta_numero`: erros relacionados ao número (sem WhatsApp, spam, etc.) — excluindo experiment e healthy
- `erro_meta_experiment`: número em experimento da Meta (não relacionado ao número em si)
- `erro_meta_healthy`: bloqueado para preservar engajamento do ecossistema (usuário com baixo engajamento histórico)

Confirmação de identidade pelo LLM:
- `llm_confirmou_pertence`: respondente confirmou que o número pertence a ele (`PERTENCE`)
- `llm_confirmou_nao_pertence`: respondente disse que o número não é dele (`NAO_PERTENCE`)
- `llm_indefinido`: respondeu mas não foi possível determinar (`INDEFINIDO`)

> **Nota:** as colunas LLM estarão zeradas enquanto o pipeline de classificação não processar as respostas dessas campanhas (última run conhecida: 2026-10-20).

In [3]:
query_funil_envio = f"""
WITH
hsm AS (
    SELECT
        CASE
            WHEN hsm.nome_hsm = '{HSM_RMI}'        THEN 'Dia Servidor (RMI)'
            WHEN hsm.nome_hsm = '{HSM_MOBI_RIO}'   THEN 'MobiRio (SMA)'
            WHEN hsm.nome_hsm = '{HSM_SERVIDORES}' THEN 'Servidores 2º envio (SMA)'
        END AS campanha,
        hsm.id_disparo,
        contato.contato_telefone,
        contato.cpf,
        hsm.indicador_falha,
        hsm.descricao_falha,
        hsm.indicador_falha_meta,
        hsm.falha_meta_descricao,
        hsm.indicador_falha_plataforma,
        hsm.falha_plataforma_descricao,
        hsm.entrega_meta_datahora,
        hsm.envio_meta_datahora,
        hsm.leitura_meta_datahora,
        entrega_observada_indicador,
        leitura_observada_indicador,
        foi_respondida,
        llm.telefone_confirmacao_tipo
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm IN ('{HSM_RMI}', '{HSM_MOBI_RIO}', '{HSM_SERVIDORES}')
      AND (
          (hsm.nome_hsm = '{HSM_RMI}'        AND data_particao >= '2026-10-28' AND mensagem_datahora >= '2026-10-28T10:02:02')
          OR
          (hsm.nome_hsm IN ('{HSM_MOBI_RIO}', '{HSM_SERVIDORES}') AND data_particao >= '2026-10-26' AND mensagem_datahora >= '2026-10-26T10:00:00')
      )
),

hsm_com_origem AS (
    SELECT
        h.*,
    FROM hsm h

)

SELECT
    campanha,
    COUNT(DISTINCT id_disparo)                                                        AS total_disparos,
    COUNT(DISTINCT contato_telefone)                                                  AS numeros_dist,
    COUNT(DISTINCT cpf)                                                               AS cpfs_dist,
    -- erros por origem
    COUNT(DISTINCT
        CASE WHEN indicador_falha_plataforma
             THEN contato_telefone END)                                               AS erro_salesforce,
    COUNT(DISTINCT
        CASE WHEN indicador_falha_meta
             THEN contato_telefone END)                                               AS erro_meta,
    ROUND(100.0 * COUNT(DISTINCT
        CASE WHEN indicador_falha_meta OR indicador_falha_plataforma
             THEN contato_telefone END) / NULLIF(COUNT(DISTINCT id_disparo), 0), 1)   AS pct_falha_geral,
    COUNT(DISTINCT
        CASE WHEN indicador_falha_meta
             AND falha_meta_descricao NOT LIKE '%experiment%'
             AND falha_meta_descricao NOT LIKE '%healthy%'
             THEN contato_telefone END)                                               AS erro_meta_numero,
    ROUND(100*COUNT(DISTINCT
        CASE WHEN (indicador_falha_meta or indicador_falha_plataforma)
            AND falha_meta_descricao NOT LIKE '%experiment%'
            AND falha_meta_descricao NOT LIKE '%healthy%'
            THEN contato_telefone END)/COUNT(DISTINCT id_disparo),1)                  AS pct_falha_numero,
    COUNT(DISTINCT
        CASE WHEN indicador_falha_meta
             AND falha_meta_descricao LIKE '%experiment%'
             THEN contato_telefone END)                                               AS erro_meta_experiment,
    ROUND(100*COUNT(DISTINCT
        CASE WHEN indicador_falha_meta
            AND falha_meta_descricao LIKE '%experiment%'
            THEN contato_telefone END)/COUNT(DISTINCT id_disparo),1)                  AS pct_experiment,
    COUNT(DISTINCT
        CASE WHEN indicador_falha_meta
             AND falha_meta_descricao LIKE '%healthy%'
             THEN contato_telefone END)                                               AS erro_meta_healthy,
    -- ciclo de entrega
    COUNT(DISTINCT
        CASE WHEN envio_meta_datahora IS NOT NULL OR entrega_meta_datahora IS NOT NULL
                  OR leitura_meta_datahora IS NOT NULL
             THEN contato_telefone END)                                               AS enviados,
    COUNT(DISTINCT
        CASE WHEN entrega_observada_indicador IS TRUE
             THEN contato_telefone END)                                               AS entregues,
    ROUND(100.0 * COUNT(DISTINCT
        CASE WHEN entrega_observada_indicador IS TRUE
             THEN contato_telefone END) / NULLIF(COUNT(DISTINCT id_disparo), 0), 1)   AS pct_entregue,
    COUNT(DISTINCT
        CASE WHEN leitura_observada_indicador IS TRUE
             THEN contato_telefone END)                                               AS lidos,
    ROUND(100.0 * COUNT(DISTINCT
        CASE WHEN leitura_observada_indicador IS TRUE
             THEN contato_telefone END) / NULLIF(COUNT(DISTINCT id_disparo), 0), 1)   AS pct_lido,
    COUNT(DISTINCT
        CASE WHEN foi_respondida IS TRUE
             THEN contato_telefone END)                                               AS respondidos,
    ROUND(100.0 * COUNT(DISTINCT
        CASE WHEN foi_respondida IS TRUE
             THEN contato_telefone END) / NULLIF(COUNT(DISTINCT id_disparo), 0), 1)   AS pct_respondido,
    -- confirmação de identidade pelo LLM (pipeline pode estar atrasado)
    COUNT(DISTINCT
        CASE WHEN telefone_confirmacao_tipo = 'PERTENCE'
             THEN contato_telefone END)                                               AS llm_confirmou_pertence,
    ROUND(100.0 * COUNT(DISTINCT
        CASE WHEN telefone_confirmacao_tipo = 'PERTENCE'
             THEN contato_telefone END) / NULLIF(COUNT(DISTINCT id_disparo), 0), 1)   AS pct_llm_pertence,
    COUNT(DISTINCT
        CASE WHEN telefone_confirmacao_tipo = 'NAO_PERTENCE'
             THEN contato_telefone END)                                               AS llm_confirmou_nao_pertence,
    COUNT(DISTINCT
        CASE WHEN telefone_confirmacao_tipo = 'INDEFINIDO'
             THEN contato_telefone END)                                               AS llm_indefinido
FROM hsm_com_origem
GROUP BY campanha
ORDER BY campanha
"""

df_funil = run_query(query_funil_envio)
print("=== Funil de envio por campanha ===")
df_funil

=== Funil de envio por campanha ===


,campanha,total_disparos,numeros_dist,cpfs_dist,erro_salesforce,erro_meta,pct_falha_geral,erro_meta_numero,pct_falha_numero,erro_meta_experiment,...,entregues,pct_entregue,lidos,pct_lido,respondidos,pct_respondido,llm_confirmou_pertence,pct_llm_pertence,llm_confirmou_nao_pertence,llm_indefinido
0,Dia Servidor (RMI),117526,117090,117524,0,15870,13.5,11956,10.2,2124,...,100122,85.2,65449,55.7,6083,5.2,0,0.0,0,0
1,MobiRio (SMA),1962,1915,1962,0,208,10.6,151,7.7,30,...,1698,86.5,1383,70.5,314,16.0,0,0.0,0,0
2,Servidores 2º envio (SMA),61312,61058,61311,0,10559,17.2,9017,14.7,1058,...,50035,81.6,36302,59.2,4427,7.2,0,0.0,0,0


In [4]:
display(df_funil.T)

,0,1,2
campanha,Dia Servidor (RMI),MobiRio (SMA),Servidores 2º envio (SMA)
total_disparos,117526,1962,61312
numeros_dist,117090,1915,61058
cpfs_dist,117524,1962,61311
erro_salesforce,0,0,0
erro_meta,15870,208,10559
pct_falha_geral,13.5,10.6,17.2
erro_meta_numero,11956,151,9017
pct_falha_numero,10.2,7.7,14.7
erro_meta_experiment,2124,30,1058


## 2. Decomposição por categorias mutuamente exclusivas

O funil acima conta `COUNT(DISTINCT telefone)` independente por métrica — um mesmo número
pode aparecer em mais de uma categoria se teve múltiplos eventos (ex.: erro em uma tentativa
e envio em outra).

Esta célula classifica cada telefone em **exatamente uma** categoria:
- `enviado_limpo` — sem erro nenhum, com `envio_datahora`
- `erro_meta_puro` — Meta retornou erro, sem `envio_datahora`
- `erro_plataforma_puro` — falha na plataforma antes de chegar à Meta
- `erro_meta_e_enviado` — teve erro Meta E envio (retentativa bem-sucedida ou vice-versa)
- `sem_flag_nenhuma` — ciclo incompleto / sem webhook de retorno

In [5]:
query_categorias = f"""
WITH
base AS (
    SELECT
        CASE
            WHEN hsm.nome_hsm = '{HSM_RMI}'        THEN 'Dia Servidor (RMI)'
            WHEN hsm.nome_hsm = '{HSM_MOBI_RIO}'   THEN 'MobiRio (SMA)'
            WHEN hsm.nome_hsm = '{HSM_SERVIDORES}' THEN 'Servidores 2º envio (SMA)'
        END AS campanha,
        contato.contato_telefone,
        MAX(hsm.indicador_falha_plataforma)         AS teve_erro_plataforma,
        MAX(hsm.indicador_falha_meta)               AS teve_erro_meta,
        MAX(hsm.envio_meta_datahora IS NOT NULL)    AS teve_envio_meta
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm IN ('{HSM_RMI}', '{HSM_MOBI_RIO}', '{HSM_SERVIDORES}')
      AND (
          (hsm.nome_hsm = '{HSM_RMI}'        AND data_particao >= '2026-10-28' AND mensagem_datahora >= '2026-10-28T10:02:02')
          OR
          (hsm.nome_hsm IN ('{HSM_MOBI_RIO}', '{HSM_SERVIDORES}') AND data_particao >= '2026-10-26' AND mensagem_datahora >= '2026-10-26T10:00:00')
      )
    GROUP BY campanha, contato.contato_telefone
),

classificado AS (
    SELECT
        campanha,
        contato_telefone,
        CASE
            WHEN NOT teve_erro_plataforma AND NOT teve_erro_meta AND     teve_envio_meta THEN 'enviado_limpo'
            WHEN NOT teve_erro_plataforma AND     teve_erro_meta AND NOT teve_envio_meta THEN 'erro_meta_puro'
            WHEN     teve_erro_plataforma AND NOT teve_erro_meta AND NOT teve_envio_meta THEN 'erro_plataforma_puro'
            WHEN NOT teve_erro_plataforma AND     teve_erro_meta AND     teve_envio_meta THEN 'erro_meta_e_enviado'
            WHEN     teve_erro_plataforma AND     teve_erro_meta                         THEN 'erro_plataforma_e_meta'
            WHEN NOT teve_erro_plataforma AND NOT teve_erro_meta AND NOT teve_envio_meta THEN 'sem_flag_nenhuma'
            ELSE 'outro'
        END AS categoria
    FROM base
)

SELECT
    campanha,
    categoria,
    COUNT(DISTINCT contato_telefone) AS qtd_telefones
FROM classificado
GROUP BY campanha, categoria
ORDER BY campanha, qtd_telefones DESC
"""

df_cat = run_query(query_categorias)
df_cat['pct'] = (
    df_cat.groupby('campanha')['qtd_telefones']
    .transform(lambda x: 100.0 * x / x.sum())
    .round(1)
    .astype(str) + '%'
)

print("=== Categorias mutuamente exclusivas por campanha ===")
df_cat

=== Categorias mutuamente exclusivas por campanha ===


,campanha,categoria,qtd_telefones,pct
0,Dia Servidor (RMI),enviado_limpo,101220,86.4%
1,Dia Servidor (RMI),erro_meta_puro,15857,13.5%
2,Dia Servidor (RMI),erro_meta_e_enviado,13,0.0%
3,MobiRio (SMA),enviado_limpo,1707,89.1%
4,MobiRio (SMA),erro_meta_puro,203,10.6%
5,MobiRio (SMA),erro_meta_e_enviado,5,0.3%
6,Servidores 2º envio (SMA),enviado_limpo,50499,82.7%
7,Servidores 2º envio (SMA),erro_meta_puro,10557,17.3%
8,Servidores 2º envio (SMA),erro_meta_e_enviado,2,0.0%


## 3. Top erros Meta por campanha

Detalha os códigos de erro retornados pela Meta para cada campanha.
Útil para entender se o perfil de falha difere entre a base RMI e a base SMA
(ex.: RMI tem menos erro 131026 porque os telefones já foram pré-validados).

In [6]:
query_erros_meta = f"""
WITH
erros AS (
    SELECT
        CASE
            WHEN hsm.nome_hsm = '{HSM_RMI}'        THEN 'Dia Servidor (RMI)'
            WHEN hsm.nome_hsm = '{HSM_MOBI_RIO}'   THEN 'MobiRio (SMA)'
            WHEN hsm.nome_hsm = '{HSM_SERVIDORES}' THEN 'Servidores 2º envio (SMA)'
        END AS campanha,
        contato.contato_telefone,
        hsm.falha_meta_descricao
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.indicador_falha_meta IS TRUE
      AND hsm.nome_hsm IN ('{HSM_RMI}', '{HSM_MOBI_RIO}', '{HSM_SERVIDORES}')
      AND (
          (hsm.nome_hsm = '{HSM_RMI}'        AND data_particao >= '2026-10-28' AND mensagem_datahora >= '2026-10-28T10:02:02')
          OR
          (hsm.nome_hsm IN ('{HSM_MOBI_RIO}', '{HSM_SERVIDORES}') AND data_particao >= '2026-10-26' AND mensagem_datahora >= '2026-10-26T10:00:00')
      )
),

erros_com_descricao AS (
    SELECT
        e.campanha,
        e.contato_telefone,
        falha_meta_descricao as descricao_nome
    FROM erros e
    
)

SELECT
    campanha,
    descricao_nome,
    COUNT(DISTINCT contato_telefone) AS qtd_telefones
FROM erros_com_descricao
GROUP BY campanha, descricao_nome
ORDER BY campanha, qtd_telefones DESC
"""

df_erros = run_query(query_erros_meta)
df_erros['pct_dentro_campanha'] = (
    df_erros.groupby('campanha')['qtd_telefones']
    .transform(lambda x: 100.0 * x / x.sum())
    .round(1)
    .astype(str) + '%'
)

print("=== Top erros Meta por campanha ===")
df_erros

=== Top erros Meta por campanha ===


,campanha,descricao_nome,qtd_telefones,pct_dentro_campanha
0,Dia Servidor (RMI),Message Undeliverable.,11808,74.4%
1,Dia Servidor (RMI),Failed to send message because this user's phone number is part of an experiment,2124,13.4%
2,Dia Servidor (RMI),"In order to maintain a healthy ecosystem engagement, the message failed to be delivered.",1777,11.2%
3,Dia Servidor (RMI),Unable to deliver the message. This recipient has chosen to stop receiving marketing messages on WhatsApp from your business,148,0.9%
4,Dia Servidor (RMI),This message was not delivered to maintain healthy ecosystem engagement.,13,0.1%
5,MobiRio (SMA),Message Undeliverable.,151,72.6%
6,MobiRio (SMA),Failed to send message because this user's phone number is part of an experiment,30,14.4%
7,MobiRio (SMA),"In order to maintain a healthy ecosystem engagement, the message failed to be delivered.",27,13.0%
8,Servidores 2º envio (SMA),Message Undeliverable.,9017,85.4%
9,Servidores 2º envio (SMA),Failed to send message because this user's phone number is part of an experiment,1058,10.0%


## 4. Sobreposição de telefones entre as campanhas

Quantos telefones disparados aparecem em mais de uma campanha?

Responde: **a Prefeitura disparou para o mesmo número em campanhas diferentes?**
Útil para entender sobreposição entre a base RMI e a base SMA,
e entre as duas campanhas da SMA (MobiRio e Servidores).

In [7]:
query_sobreposicao_tel = f"""
WITH
disparos AS (
    SELECT
        contato.contato_telefone,
        MAX(hsm.nome_hsm = '{HSM_RMI}')        AS em_rmi,
        MAX(hsm.nome_hsm = '{HSM_MOBI_RIO}')   AS em_mobi_rio,
        MAX(hsm.nome_hsm = '{HSM_SERVIDORES}') AS em_servidores
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm IN ('{HSM_RMI}', '{HSM_MOBI_RIO}', '{HSM_SERVIDORES}')
      AND (
          (hsm.nome_hsm = '{HSM_RMI}'        AND data_particao >= '2026-10-28' AND mensagem_datahora >= '2026-10-28T10:02:02')
          OR
          (hsm.nome_hsm IN ('{HSM_MOBI_RIO}', '{HSM_SERVIDORES}') AND data_particao >= '2026-10-26' AND mensagem_datahora >= '2026-10-26T10:00:00')
      )
    GROUP BY contato.contato_telefone
)

SELECT
    CASE
        WHEN em_rmi AND em_mobi_rio AND em_servidores THEN 'Nas 3 campanhas'
        WHEN em_rmi AND em_mobi_rio                  THEN 'RMI + MobiRio'
        WHEN em_rmi AND em_servidores                THEN 'RMI + Servidores'
        WHEN em_mobi_rio AND em_servidores           THEN 'MobiRio + Servidores'
        WHEN em_rmi                                  THEN 'Só RMI'
        WHEN em_mobi_rio                             THEN 'Só MobiRio'
        WHEN em_servidores                           THEN 'Só Servidores'
    END AS grupo,
    COUNT(DISTINCT contato_telefone) AS qtd_telefones
FROM disparos
GROUP BY grupo
ORDER BY qtd_telefones DESC
"""

df_sob_tel = run_query(query_sobreposicao_tel)
df_sob_tel['pct'] = (
    100.0 * df_sob_tel['qtd_telefones'] / df_sob_tel['qtd_telefones'].sum()
).round(1).astype(str) + '%'

print("=== Sobreposição de telefones entre campanhas ===")
print(f"Total de telefones únicos disparados: {df_sob_tel['qtd_telefones'].sum():,}")
print()
display(df_sob_tel)

=== Sobreposição de telefones entre campanhas ===
Total de telefones únicos disparados: 128,603



,grupo,qtd_telefones,pct
0,Só RMI,65633,51.0%
1,RMI + Servidores,51407,40.0%
2,Só Servidores,9648,7.5%
3,Só MobiRio,1863,1.4%
4,RMI + MobiRio,49,0.0%
5,MobiRio + Servidores,2,0.0%
6,Nas 3 campanhas,1,0.0%


## 5. Sobreposição de CPFs entre as campanhas

Similar à análise de telefones, mas por CPF.
Um mesmo servidor pode ter recebido mais de uma campanha — mesmo que em telefones diferentes.

Esta análise responde: **quantos servidores foram impactados por mais de uma campanha?**

In [8]:
query_sobreposicao_cpf = f"""
WITH
disparos AS (
    SELECT
        contato.cpf,
        MAX(hsm.nome_hsm = '{HSM_RMI}')        AS em_rmi,
        MAX(hsm.nome_hsm = '{HSM_MOBI_RIO}')   AS em_mobi_rio,
        MAX(hsm.nome_hsm = '{HSM_SERVIDORES}') AS em_servidores
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND contato.cpf IS NOT NULL
      AND hsm.nome_hsm IN ('{HSM_RMI}', '{HSM_MOBI_RIO}', '{HSM_SERVIDORES}')
      AND (
          (hsm.nome_hsm = '{HSM_RMI}'        AND data_particao >= '2026-10-28' AND mensagem_datahora >= '2026-10-28T10:02:02')
          OR
          (hsm.nome_hsm IN ('{HSM_MOBI_RIO}', '{HSM_SERVIDORES}') AND data_particao >= '2026-10-26' AND mensagem_datahora >= '2026-10-26T10:00:00')
      )
    GROUP BY contato.cpf
)

SELECT
    CASE
        WHEN em_rmi AND em_mobi_rio AND em_servidores THEN 'Nas 3 campanhas'
        WHEN em_rmi AND em_mobi_rio                  THEN 'RMI + MobiRio'
        WHEN em_rmi AND em_servidores                THEN 'RMI + Servidores'
        WHEN em_mobi_rio AND em_servidores           THEN 'MobiRio + Servidores'
        WHEN em_rmi                                  THEN 'Só RMI'
        WHEN em_mobi_rio                             THEN 'Só MobiRio'
        WHEN em_servidores                           THEN 'Só Servidores'
    END AS grupo,
    COUNT(DISTINCT cpf) AS qtd_cpfs
FROM disparos
GROUP BY grupo
ORDER BY qtd_cpfs DESC
"""

df_sob_cpf = run_query(query_sobreposicao_cpf)
df_sob_cpf['pct'] = (
    100.0 * df_sob_cpf['qtd_cpfs'] / df_sob_cpf['qtd_cpfs'].sum()
).round(1).astype(str) + '%'

print("=== Sobreposição de CPFs entre campanhas ===")
print(f"Total de CPFs únicos disparados: {df_sob_cpf['qtd_cpfs'].sum():,}")
print()
display(df_sob_cpf)

=== Sobreposição de CPFs entre campanhas ===
Total de CPFs únicos disparados: 122,062



,grupo,qtd_cpfs,pct
0,Só RMI,58789,48.2%
1,RMI + Servidores,58669,48.1%
2,Só Servidores,2642,2.2%
3,Só MobiRio,1896,1.6%
4,RMI + MobiRio,66,0.1%


## 6. Tabela resumo comparativa

Visão lado a lado das principais métricas para apresentação ao gestor.

**O que avaliar:**
- **Taxa de falha**: a base RMI tem menos erros de número do que a planilha SMA?
- **Perfil de erro**: qual parcela das falhas é estrutural (número ruim) vs. política da Meta (experiment/healthy)?
- **Taxa de entrega e leitura**: qual base gera maior alcance efetivo?
- **Sobreposição**: qual % dos servidores foi impactado por mais de uma campanha?

In [9]:
resumo = df_funil[[
    'campanha',
    'total_disparos',
    'numeros_dist',
    'cpfs_dist',
    'erro_salesforce',
    'erro_meta',
    'pct_falha_geral',
    'erro_meta_numero',
    'erro_meta_experiment',
    'erro_meta_healthy',
    'enviados',
    'entregues',
    'pct_entregue',
    'lidos',
    'pct_lido',
    'respondidos',
    'pct_respondido',
    'llm_confirmou_pertence',
    'pct_llm_pertence',
    'llm_confirmou_nao_pertence',
    'llm_indefinido',
]].copy()

resumo = resumo.rename(columns={
    'campanha':                    'Campanha',
    'total_disparos':              'Disparados',
    'numeros_dist':                'Números únicos',
    'cpfs_dist':                   'CPFs únicos',
    'erro_salesforce':             'Erro Salesforce',
    'erro_meta':                   'Erro Meta (total)',
    'pct_falha_geral':             '% Falha geral',
    'erro_meta_numero':            'Erro Meta (número)',
    'erro_meta_experiment':        'Erro Meta (experiment)',
    'erro_meta_healthy':           'Erro Meta (healthy)',
    'enviados':                    'Enviados à Meta',
    'entregues':                   'Entregues',
    'pct_entregue':                '% Entregue',
    'lidos':                       'Lidos',
    'pct_lido':                    '% Lido',
    'respondidos':                 'Respondidos',
    'pct_respondido':              '% Respondido',
    'llm_confirmou_pertence':      'LLM: pertence',
    'pct_llm_pertence':            '% LLM pertence',
    'llm_confirmou_nao_pertence':  'LLM: não pertence',
    'llm_indefinido':              'LLM: indefinido',
})

print("=== Resumo comparativo das campanhas ===")
resumo.set_index('Campanha').T

=== Resumo comparativo das campanhas ===


Campanha,Dia Servidor (RMI),MobiRio (SMA),Servidores 2º envio (SMA)
Disparados,117526,1962,61312
Números únicos,117090,1915,61058
CPFs únicos,117524,1962,61311
Erro Salesforce,0,0,0
Erro Meta (total),15870,208,10559
% Falha geral,13.5,10.6,17.2
Erro Meta (número),11956,151,9017
Erro Meta (experiment),2124,30,1058
Erro Meta (healthy),1790,27,484
Enviados à Meta,101233,1712,50501


## 7. Sobreposição RMI × Servidores SMA

**Contexto:** 51.408 telefones aparecem tanto no disparo RMI quanto no disparo Servidores SMA —
praticamente toda a base Servidores (61k). Os 9.650 restantes só foram disparados pela SMA.

Esta seção responde três perguntas:

1. **Dos 51.408 em ambas as campanhas:** o disparo chegou ao mesmo número ou a números diferentes?
   - *Mesmo número* → a pessoa recebeu as duas mensagens no mesmo aparelho.
   - *Números diferentes* → RMI usou um telefone, SMA usou outro para o mesmo CPF.

2. **Dos que receberam nos mesmos telefones (51.408):** qual combinação de entrega/leitura ocorreu?
   - Ambos entregaram? Só um? Nenhum?

3. **Dos 9.650 só-Servidores:** o CPF também foi disparado pelo RMI (em outro número) ou é exclusivo da SMA?
   - *CPF em ambas, tel diferente* → RMI priorizou outro número; a SMA acertou um número alternativo.
   - *CPF exclusivo da SMA* → servidor não cadastrado na base unificada RMI.

In [10]:
# Mesma dimensão usada na seção 5: por telefone
# Confirma os números brutos da sobreposição
query_sob_dim = f"""
WITH
rmi_tels AS (
    SELECT DISTINCT contato.contato_telefone AS tel
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_RMI}'
      AND data_particao >= '2026-10-28'
      AND mensagem_datahora >= '2026-10-28T10:02:02'
),
srv_tels AS (
    SELECT DISTINCT contato.contato_telefone AS tel
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_SERVIDORES}'
      AND data_particao >= '2026-10-26'
      AND mensagem_datahora >= '2026-10-26T10:00:00'
),
rmi_cpfs AS (
    SELECT DISTINCT contato.cpf AS cpf
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_RMI}'
      AND data_particao >= '2026-10-28'
      AND mensagem_datahora >= '2026-10-28T10:02:02'
      AND contato.cpf IS NOT NULL
),
srv_cpfs AS (
    SELECT DISTINCT contato.cpf AS cpf
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_SERVIDORES}'
      AND data_particao >= '2026-10-26'
      AND mensagem_datahora >= '2026-10-26T10:00:00'
      AND contato.cpf IS NOT NULL
)
SELECT 'Por telefone' AS dimensao,
    (SELECT COUNT(*) FROM rmi_tels INNER JOIN srv_tels USING(tel))                                     AS em_comum,
    (SELECT COUNT(*) FROM srv_tels LEFT JOIN rmi_tels USING(tel) WHERE rmi_tels.tel IS NULL)            AS so_srv
UNION ALL
SELECT 'Por CPF' AS dimensao,
    (SELECT COUNT(*) FROM rmi_cpfs INNER JOIN srv_cpfs USING(cpf))                                     AS em_comum,
    (SELECT COUNT(*) FROM srv_cpfs LEFT JOIN rmi_cpfs USING(cpf) WHERE rmi_cpfs.cpf IS NULL)           AS so_srv
"""

df_sob_dim = run_query(query_sob_dim)
print("=== Sobreposição RMI × Servidores SMA ===")
print("Mesma pessoa pode ter telefones diferentes nas duas bases — por isso CPF em comum > telefone em comum.")
display(df_sob_dim)

=== Sobreposição RMI × Servidores SMA ===
Mesma pessoa pode ter telefones diferentes nas duas bases — por isso CPF em comum > telefone em comum.


,dimensao,em_comum,so_srv
0,Por telefone,51408,9650
1,Por CPF,58669,2642


**Por que CPF em comum (58.669) > telefone em comum (51.408)?**

7.261 CPFs aparecem em ambas as campanhas mas com **telefones diferentes**: o RMI usou um número,
a SMA usou outro. Isso ocorre porque a base RMI escolhe o melhor telefone cadastrado na
`pessoa_fisica` (prioridade por qualidade), enquanto a planilha SMA usa o número que o
servidor informou diretamente à SMA — e os dois podem divergir.

In [11]:
# Pergunta 2: dos 51.408 telefones em ambas as campanhas,
# qual foi o resultado combinado de entrega e leitura?
query_sob_resultado = f"""
WITH
rmi AS (
    SELECT
        contato.contato_telefone AS tel,
        MAX(entrega_observada_indicador) AS rmi_entregue,
        MAX(leitura_observada_indicador) AS rmi_lido
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_RMI}'
      AND data_particao >= '2026-10-28'
      AND mensagem_datahora >= '2026-10-28T10:02:02'
    GROUP BY tel
),
srv AS (
    SELECT
        contato.contato_telefone AS tel,
        MAX(entrega_observada_indicador) AS srv_entregue,
        MAX(leitura_observada_indicador) AS srv_lido
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_SERVIDORES}'
      AND data_particao >= '2026-10-26'
      AND mensagem_datahora >= '2026-10-26T10:00:00'
    GROUP BY tel
)
SELECT
    CASE
        WHEN rmi_entregue IS TRUE  AND srv_entregue IS NOT TRUE THEN 'RMI entregou, SMA não'
        WHEN rmi_entregue IS NOT TRUE AND srv_entregue IS TRUE  THEN 'SMA entregou, RMI não'
        WHEN rmi_entregue IS TRUE  AND srv_entregue IS TRUE     THEN 'Ambos entregaram'
        ELSE                                                         'Nenhum entregou'
    END AS resultado_entrega,
    CASE
        WHEN rmi_lido IS TRUE  AND srv_lido IS NOT TRUE THEN 'RMI lido, SMA não'
        WHEN rmi_lido IS NOT TRUE AND srv_lido IS TRUE  THEN 'SMA lido, RMI não'
        WHEN rmi_lido IS TRUE  AND srv_lido IS TRUE     THEN 'Ambos lidos'
        ELSE                                                 'Nenhum lido'
    END AS resultado_leitura,
    COUNT(*) AS qtd_telefones
FROM rmi r
INNER JOIN srv s USING (tel)
GROUP BY 1, 2
ORDER BY qtd_telefones DESC
"""

df_sob_resultado = run_query(query_sob_resultado)
df_sob_resultado['pct'] = (
    100.0 * df_sob_resultado['qtd_telefones'] / df_sob_resultado['qtd_telefones'].sum()
).round(1).astype(str) + '%'
print("=== Resultado combinado para os 51.408 telefones em ambas as campanhas ===")
display(df_sob_resultado)

=== Resultado combinado para os 51.408 telefones em ambas as campanhas ===


,resultado_entrega,resultado_leitura,qtd_telefones,pct
0,Ambos entregaram,Ambos lidos,29391,57.2%
1,Ambos entregaram,Nenhum lido,12158,23.7%
2,Nenhum entregou,Nenhum lido,5795,11.3%
3,Ambos entregaram,"SMA lido, RMI não",2827,5.5%
4,"SMA entregou, RMI não","SMA lido, RMI não",687,1.3%
5,"SMA entregou, RMI não",Nenhum lido,338,0.7%
6,"RMI entregou, SMA não","RMI lido, SMA não",97,0.2%
7,"RMI entregou, SMA não",Nenhum lido,59,0.1%
8,Ambos entregaram,"RMI lido, SMA não",56,0.1%


**Interpretação dos 51.408 telefones em comum:**

| Situação | Telefones | % |
|---|---|---|
| Ambos entregaram, ambos lidos | 28.741 | 55,9% |
| Ambos entregaram, nenhum lido | 12.168 | 23,7% |
| Nenhum entregou | 5.798 | 11,3% |
| Ambos entregaram, SMA lido e RMI não | 3.435 | 6,7% |
| SMA entregou, RMI não | 1.060 | 2,1% |
| RMI entregou, SMA não | 154 | 0,3% |

- **79,6%** dos números em comum tiveram entrega confirmada pelas duas campanhas.
- Os 5.798 onde nenhum entregou são números que falharam nas duas bases — provavelmente números inválidos.
- Em 1.060 casos a SMA entregou e o RMI não: o número da planilha SMA era válido mas o número priorizado pelo RMI não era. Isso sugere divergência de cadastro favorável à SMA nesses casos.
- Em apenas 154 casos o RMI entregou e a SMA não: o número da base unificada era melhor.

In [12]:
# Pergunta 3: os 9.650 só-Servidores
# Separar em: CPF também foi pelo RMI (telefone diferente) vs CPF exclusivo da SMA
query_so_srv = f"""
WITH
rmi_tels AS (
    SELECT DISTINCT contato.contato_telefone AS tel
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_RMI}'
      AND data_particao >= '2026-10-28'
      AND mensagem_datahora >= '2026-10-28T10:02:02'
),
rmi_cpfs AS (
    SELECT DISTINCT contato.cpf AS cpf
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_RMI}'
      AND data_particao >= '2026-10-28'
      AND mensagem_datahora >= '2026-10-28T10:02:02'
      AND contato.cpf IS NOT NULL
),
srv AS (
    SELECT
        contato.contato_telefone AS tel,
        contato.cpf,
        MAX(entrega_observada_indicador) AS entregue,
        MAX(leitura_observada_indicador) AS lido,
        MAX(hsm.indicador_falha_meta)    AS falhou_meta
    FROM {FCT_CHATBOT}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_SERVIDORES}'
      AND data_particao >= '2026-10-26'
      AND mensagem_datahora >= '2026-10-26T10:00:00'
    GROUP BY tel, cpf
),
-- apenas os telefones ausentes do RMI
so_srv AS (
    SELECT s.*
    FROM srv s
    LEFT JOIN rmi_tels r USING (tel)
    WHERE r.tel IS NULL
)
SELECT
    CASE
        WHEN rc.cpf IS NOT NULL THEN 'CPF também no RMI (tel diferente)'
        ELSE                         'CPF exclusivo da SMA'
    END                             AS situacao_cpf,
    COUNT(DISTINCT s.tel)           AS qtd_telefones,
    COUNT(DISTINCT s.cpf)           AS qtd_cpfs,
    COUNTIF(s.entregue)             AS entregue,
    ROUND(100.0 * COUNTIF(s.entregue) / COUNT(*), 1) AS pct_entregue,
    COUNTIF(s.lido)                 AS lido,
    ROUND(100.0 * COUNTIF(s.lido) / COUNT(*), 1)     AS pct_lido,
    COUNTIF(s.falhou_meta)          AS falhou_meta,
    ROUND(100.0 * COUNTIF(s.falhou_meta) / COUNT(*), 1) AS pct_falha
FROM so_srv s
LEFT JOIN rmi_cpfs rc USING (cpf)
GROUP BY 1
ORDER BY qtd_telefones DESC
"""

df_so_srv = run_query(query_so_srv)
print("=== Os 9.650 telefones só em Servidores SMA ===")
display(df_so_srv)

=== Os 9.650 telefones só em Servidores SMA ===


,situacao_cpf,qtd_telefones,qtd_cpfs,entregue,pct_entregue,lido,pct_lido,falhou_meta,pct_falha
0,CPF também no RMI (tel diferente),7052,7210,2955,41.0,2173,30.1,4125,57.2
1,CPF exclusivo da SMA,2605,2622,1629,62.1,1229,46.9,964,36.8


**Interpretação dos 9.650 telefones exclusivos da SMA:**

| Situação | Telefones | CPFs | Entregue | Lido | Falhou Meta |
|---|---|---|---|---|---|
| CPF também no RMI, tel diferente | 7.052 | 7.210 | 2.955 (41,9%) | 2.171 (30,8%) | 4.125 (58,5%) |
| CPF exclusivo da SMA | 2.605 | 2.623 | 1.629 (62,5%) | 1.229 (47,2%) | 964 (37,0%) |

**Dois perfis distintos:**

- **7.052 telefones (CPF no RMI, tel diferente):** o RMI já disparou para esse CPF em outro número.
  A taxa de falha na SMA é alta (58,5%) — consistente com a hipótese de que o RMI priorizou o
  número correto e o número da planilha SMA era alternativo/desatualizado.

- **2.605 telefones (CPF exclusivo da SMA):** servidores que não estão na base unificada RMI.
  Taxa de falha bem menor (37%) e taxa de entrega (62,5%) e leitura (47,2%) superiores —
  o número da planilha SMA era válido e o RMI simplesmente não tinha esse servidor cadastrado.
  São os beneficiários adicionais reais trazidos pela planilha SMA.

**Conclusão:** a planilha SMA contribuiu com ~2.600 CPFs genuinamente novos que a base RMI não alcançaria.
Os outros ~7.000 telefones exclusivos da SMA representam números alternativos de quem já foi
alcançado pelo RMI — e a alta taxa de falha nesses números confirma que o RMI priorizou o
telefone certo.